In [1]:
import pandas as pd
import numpy as np
import requests

#### 1. Прочитать json файл, отобразить числа с двумя знаками после запятой

In [2]:
df = pd.read_json('../data/auto.json')
pd.options.display.float_format = '{:.2f}'.format
df

,CarNumber,Refund,Fines,Make,Model
0,Y163O8161RUS,2,3200.00,Ford,Focus
1,E432XX77RUS,1,6500.00,Toyota,Camry
2,7184TT36RUS,1,2100.00,Ford,Focus
3,X582HE161RUS,2,2000.00,Ford,Focus
4,92918M178RUS,1,5700.00,Ford,Focus
...,...,...,...,...,...
720,Y163O8161RUS,2,1600.00,Ford,Focus
721,M0309X197RUS,1,22300.00,Ford,Focus
722,O673E8197RUS,2,600.00,Ford,Focus
723,8610T8154RUS,1,2000.00,Ford,Focus


#### 2. Создать сэмпл из 200 новых наблюдений, присвоить существующим машинам (без новых комбинаций номеров, марок и моделей) рандомные штрафы и оплату, добавить к базе данных (конкатенировать)

In [3]:
unique_cars = df.drop_duplicates(subset=['CarNumber', 'Make', 'Model'])
sample_cars = unique_cars.sample(n=200, random_state=21)

# Берём случайые Refund и Fines из существующих значений
random_refunds = df['Refund'].sample(n=200, random_state=21, replace=True).values
random_fines = df['Fines'].sample(n=200, random_state=21, replace=True).values

sample_cars['Refund'] = random_refunds
sample_cars['Fines'] = random_fines

# Конкатенируем с оригинальным DataFrame
concat_rows = pd.concat([df, sample_cars], ignore_index=True)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 725 entries, 0 to 724
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  725 non-null    str    
 1   Refund     725 non-null    int64  
 2   Fines      725 non-null    float64
 3   Make       725 non-null    str    
 4   Model      725 non-null    str    
dtypes: float64(1), int64(1), str(3)
memory usage: 28.4 KB


In [5]:
concat_rows.info()

<class 'pandas.DataFrame'>
RangeIndex: 925 entries, 0 to 924
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   CarNumber  925 non-null    str    
 1   Refund     925 non-null    int64  
 2   Fines      925 non-null    float64
 3   Make       925 non-null    str    
 4   Model      925 non-null    str    
dtypes: float64(1), int64(1), str(3)
memory usage: 36.3 KB


#### 3. Обогатить concat_rows новой колонкой "Year"

In [6]:
np.random.seed(21)
years = pd.Series(np.random.randint(1980, 2020, size=len(concat_rows)), name='Year')

fines = pd.concat([concat_rows, years], axis=1)
fines.head(10)

,CarNumber,Refund,Fines,Make,Model,Year
0,Y163O8161RUS,2,3200.00,Ford,Focus,1989
1,E432XX77RUS,1,6500.00,Toyota,Camry,1995
2,7184TT36RUS,1,2100.00,Ford,Focus,1984
3,X582HE161RUS,2,2000.00,Ford,Focus,2015
4,92918M178RUS,1,5700.00,Ford,Focus,2014
5,H234YH197RUS,2,6000.00,Ford,Focus,1990
6,E40577152RUS,1,8594.59,Ford,Focus,1988
7,707987163RUS,2,2200.00,Ford,Focus,2016
8,K330T8197RUS,2,8200.00,Skoda,Octavia,2018
9,X786CO96RUS,1,8594.59,Ford,Focus,2000


#### 4. Обогатить из другого DataFrame

In [7]:
df_surnames = pd.read_json('../../datasets/surname.json')
df_surnames.sample(8)

,0,1,2
53,MILLER,1161437,7
98,WOOD,250715,84
4,ANDERSON,784404,15
100,YOUNG,484447,32
8,BROOKS,251663,82
54,MITCHELL,384486,48
42,JONES,1425470,5
79,ROGERS,302261,66


In [8]:
df_surnames.describe()

,0,1,2
count,101,101,101
unique,101,101,101
top,NAME,COUNT,RANK
freq,1,1,1


In [9]:
df_surnames.columns = ['NAME', 'COUNT', 'RANK']
surnames = df_surnames['NAME'].astype(str).str.replace(r'[, \[/]{}\"]+', '', regex=True)
unique_car_numbers = fines['CarNumber'].unique()
surnames_series = pd.Series(surnames.values, name='Surname')
surnames_series

0          NAME
1         ADAMS
2         ALLEN
3       ALVAREZ
4      ANDERSON
         ...   
96     WILLIAMS
97       WILSON
98         WOOD
99       WRIGHT
100       YOUNG
Name: Surname, Length: 101, dtype: str

In [10]:
owners = pd.DataFrame({
    'CarNumber': unique_car_numbers,
    'Surname': surnames_series.sample(n=len(unique_car_numbers), replace=True, random_state=21)
})
owners

,CarNumber,Surname
73,Y163O8161RUS,REYES
79,E432XX77RUS,ROGERS
56,7184TT36RUS,MORALES
4,X582HE161RUS,ANDERSON
48,92918M178RUS,LONG
...,...,...
72,O136HO197RUS,REED
53,O22097197RUS,MILLER
91,M0309X197RUS,TURNER
8,O673E8197RUS,BROOKS


In [11]:
# 5 новых строк в fines
new_rows_fines = pd.DataFrame({
    'CarNumber': ['AAA7898RUS', 'BBBS4589RUS', 'RYTU56887RUS', 'WWFS4711RUS', 'XXX3333RUS'],
    'Refund': [2, 1, 2, 1, 2],
    'Fines': [1000, 2000, 8000, 10000, 15000],
    'Make': ['Toyota', 'Kia', 'Daewoo', 'Mazda', 'Renault'],
    'Model': ['Mark2', 'Rio', 'Uz07', 'Ex-ve', 'Duster'],
    'Year': [1992, 2015, 2007, 2018, 2013]
})

fines = pd.concat([fines, new_rows_fines], ignore_index=True)
fines.tail()

,CarNumber,Refund,Fines,Make,Model,Year
925,AAA7898RUS,2,1000.00,Toyota,Mark2,1992
926,BBBS4589RUS,1,2000.00,Kia,Rio,2015
927,RYTU56887RUS,2,8000.00,Daewoo,Uz07,2007
928,WWFS4711RUS,1,10000.00,Mazda,Ex-ve,2018
929,XXX3333RUS,2,15000.00,Renault,Duster,2013


In [12]:
# 3 новых строки в owners
new_rows_owners = pd.DataFrame({
    'CarNumber': ['HJGT009BLR', 'LLLL002BLR', 'RTRTR003BLR'],
    'Surname': ['POPOV', 'VETROV', 'DZYUBA']
})

owners = pd.concat([owners.iloc[:-20], new_rows_owners], ignore_index=True)
owners.tail()

,CarNumber,Surname
509,O50197197RUS,LEE
510,7608EE777RUS,HARRIS
511,HJGT009BLR,POPOV
512,LLLL002BLR,VETROV
513,RTRTR003BLR,DZYUBA


In [13]:
# Только номера, которые есть в обоих (inner)
result_inner = fines.merge(owners, on='CarNumber', how='inner')
# Все номера (outer)
result_outer = fines.merge(owners, on='CarNumber', how='outer')
# Только номера из fines (left)
result_left = fines.merge(owners, on='CarNumber', how='left')
# Только номера из owners (right)
result_right = fines.merge(owners, on='CarNumber', how='right')

#### 5. Сводная таблица (pivot table) из fines

In [14]:
pivot_table = pd.pivot_table(
    fines,
    values='Fines',
    index=['Make', 'Model'],
    columns='Year',
    aggfunc='sum'
)

print(pivot_table)

Year                   1980      1981      1982     1983      1984      1985  \
Make       Model                                                               
Audi       None         NaN       NaN       NaN      NaN       NaN       NaN   
BMW        None         NaN       NaN       NaN      NaN       NaN       NaN   
Daewoo     Uz07         NaN       NaN       NaN      NaN       NaN       NaN   
Ford       Focus   65794.59 396189.17 140383.76 62600.00 105994.59 189583.76   
           Mondeo       NaN       NaN       NaN      NaN       NaN       NaN   
Kia        Rio          NaN       NaN       NaN      NaN       NaN       NaN   
Mazda      Ex-ve        NaN       NaN       NaN      NaN       NaN       NaN   
Renault    Duster       NaN       NaN       NaN      NaN       NaN       NaN   
Skoda      Octavia  5000.00   2400.00   6900.00 12094.59   6500.00  10294.59   
Toyota     Camry   14000.00   8594.59       NaN  7200.00       NaN       NaN   
           Corolla      NaN       NaN   

In [15]:
fines.to_csv('../data/fines.csv', index=False)
owners.to_csv('../data/owners.csv', index=False)